In [1]:
!pip install pandas requests tqdm


[notice] A new release of pip is available: 23.2.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import requests
from tqdm import tqdm
import os

In [3]:
# Task 1: JSON formatting
PROMPT_TEMPLATE_1 = """
Instruction: Generate ONLY a JSON output following this schema:
{{ 
  "review": "<review>" 
}}

Input:
{content}

Expected output (only JSON, no extra text):
"""


In [4]:
# Task 2: JSON + Translation
PROMPT_TEMPLATE_2 = """Instruction: Translate the following review text from English to Italian and format the response in JSON according to the specified schema.

**Required JSON Format:** Ensure the response is formatted in JSON according to the following schema:

{{
  "review": "<original_review>",
  "translate": "<italian_translation>"
}}

Example:

"I recently visited the restaurant 'La Dolce Vita' in Rome and was thrilled with the service and food. The waiter, Marco, was exceptionally friendly and the truffle risotto was simply divine. I can't wait to return and recommend this place to my friends."

```json
{{
  "review": "I recently visited the restaurant 'La Dolce Vita' in Rome and was thrilled with the service and food. The waiter, Marco, was exceptionally friendly and the truffle risotto was simply divine. I can't wait to return and recommend this place to my friends.",
  "translate": "Ho recentemente visitato il ristorante 'La Dolce Vita' a Roma e sono rimasto entusiasta del servizio e del cibo. Il cameriere, Marco, è stato eccezionalmente cordiale e il risotto al tartufo era semplicemente divino. Non vedo l'ora di tornare e raccomandare questo posto ai miei amici."
}}
```

{content}"""

In [5]:
# Task 3: JSON + Translation + Sentiment Analysis
PROMPT_TEMPLATE_3 = """Instruction: Analyze the following review text and provide the outputs formatted in JSON:

1. **Translation:** Translate the review from English to Italian.
2. **Sentiment Classification:** Indicate whether the sentiment of the review is "positive" or "negative".
3. **Required JSON Format:** Ensure the response is formatted in JSON according to the following schema:

{{
  "review": "<original_review>",
  "translate": "<italian_translation>",
  "sentiment": "<sentiment>"
}}

Example:

"I recently visited the restaurant 'La Dolce Vita' in Rome and was thrilled with the service and food. The waiter, Marco, was exceptionally friendly and the truffle risotto was simply divine. I can't wait to return and recommend this place to my friends."

```json
{{
  "review": "I recently visited the restaurant 'La Dolce Vita' in Rome and was thrilled with the service and food. The waiter, Marco, was exceptionally friendly and the truffle risotto was simply divine. I can't wait to return and recommend this place to my friends.",
  "translate": "Ho recentemente visitato il ristorante 'La Dolce Vita' a Roma e sono rimasto entusiasta del servizio e del cibo. Il cameriere, Marco, è stato eccezionalmente cordiale e il risotto al tartufo era semplicemente divino. Non vedo l'ora di tornare e raccomandare questo posto ai miei amici.",
  "sentiment": "positive"
}}
```

{content}"""

In [6]:
# Task 4: JSON + Traduzione + Sentiment Analysis + NER 
PROMPT_TEMPLATE_4 = """Instruction: Analyze the following review text and provide the outputs formatted in JSON:

1. **Translation:** Translate the review from English to Italian.
2. **Sentiment Classification:** Indicate whether the sentiment of the review is "positive" or "negative".
3. **Named Entity Extraction:** List all named entities present in the text, categorizing them by label (PERSON, ORG, LOC).
4. **Required JSON Format:** Ensure the response is formatted in JSON according to the following schema:

{{
  "review": "<original_review>",
  "translate": "<italian_translation>",
  "sentiment": "<sentiment>",
  "entities": [
    {{
      "label": "<label>",
      "value": "<value>"
    }}
  ]
}}

Example:

"I recently visited the restaurant 'La Dolce Vita' in Rome and was thrilled with the service and food. The waiter, Marco, was exceptionally friendly and the truffle risotto was simply divine. I can't wait to return and recommend this place to my friends."

```json
{{
  "review": "I recently visited the restaurant 'La Dolce Vita' in Rome and was thrilled with the service and food. The waiter, Marco, was exceptionally friendly and the truffle risotto was simply divine. I can't wait to return and recommend this place to my friends.",
  "translate": "Ho recentemente visitato il ristorante 'La Dolce Vita' a Roma e sono rimasto entusiasta del servizio e del cibo. Il cameriere, Marco, è stato eccezionalmente cordiale e il risotto al tartufo era semplicemente divino. Non vedo l'ora di tornare e raccomandare questo posto ai miei amici.",
  "sentiment": "positive",
  "entities": [
    {{
      "label": "ORG",
      "value": "La Dolce Vita"
    }},
    {{
      "label": "LOC",
      "value": "Rome"
    }},
    {{
      "label": "PERSON",
      "value": "Marco"
    }}
  ]
}}
```

{content}"""


In [7]:
# Dictionary mapping task numbers to prompt templates
PROMPT_TEMPLATES = {
    1: PROMPT_TEMPLATE_1,
    2: PROMPT_TEMPLATE_2,
    3: PROMPT_TEMPLATE_3,
    4: PROMPT_TEMPLATE_4
}

TASK_NAMES = {
    1: "json_only",
    2: "json_translation", 
    3: "json_translation_sentiment",
    4: "json_translation_sentiment_ner"
}

# Define the model name to use
MODEL_NAME = "qwen3:4b-instruct"

# Define the input file path
INPUT_FILE_PATH = "../resources/IMDB Dataset 500 Sampled With Translate.csv"

print(f"✓ Configurazione completata:")
print(f"  - Modello: {MODEL_NAME}")
print(f"  - File input: {INPUT_FILE_PATH}")
print(f"  - Task configurati: {len(TASK_NAMES)}")

✓ Configurazione completata:
  - Modello: qwen3:4b-instruct
  - File input: ../resources/IMDB Dataset 500 Sampled With Translate.csv
  - Task configurati: 4


In [8]:
def process_review(review: str, model_name: str, prompt_template: str) -> str:
    """
    It processes the review text and returns the LLM response as string.
    
    Arguments:
        review (str): The review text.
        model_name (str): The model name for Ollama.
        prompt_template (str): The prompt template to use.
        
    Return:
        The LLM response as string.
    """
    try:
        return requests.post(
            url="http://localhost:11434/api/generate",
            json={
                "model": model_name,
                "prompt": prompt_template.format(content=review),
                "stream": False
            }
        ).json()["response"]
    except Exception as e:
        print(f"Error invoking the chain: {str(e)}")
        return None

In [9]:
def call_model_llm(model_name: str, task_number: int, input_file_path: str) -> None:
    """
    It calls the LLM model using Ollama for a specific task. 
    
    Arguments:
        model_name: The name of the model to invoke via Ollama.
        task_number: Which task to run (1-4).
        input_file_path: Path to the input CSV file.
    
    Return:
        None (saves results to file).
    """
    # Get the appropriate prompt template
    prompt_template = PROMPT_TEMPLATES[task_number]
    task_name = TASK_NAMES[task_number]
    
    # Define output file path
    output_file_path = f"../resources/sampled_reviews_task_{task_number}_{task_name}_{model_name.replace(':', '_')}.csv"
    
    # Check if output file exists, if not create it
    if not os.path.exists(output_file_path):
        sampled = pd.read_csv(input_file_path)
        sampled["output"] = sampled.apply(lambda row: "$$$", axis=1)
        sampled.to_csv(output_file_path, index=False)
        print(f"✓ Created new output file: {output_file_path}")
    else:
        print(f"✓ Using existing output file: {output_file_path}")
    
    # Load the dataframe
    dataframe = pd.read_csv(output_file_path)
    already_done_part = dataframe[~(dataframe.output == "$$$")].copy()
    slice_to_work_on = dataframe[dataframe.output == "$$$"].copy()
    slice_to_work_on.reset_index(inplace=True, drop=True)
    total_rows = len(slice_to_work_on)
    
    print(f"Processing Task {task_number} ({task_name}) with {total_rows} reviews...")
    
    for i in tqdm(range(total_rows), total=total_rows, desc=f"Task {task_number}"):
        row = slice_to_work_on.iloc[i]
        result = process_review(row["review"], model_name, prompt_template)
        slice_to_work_on.loc[i, "output"] = result
        updated_df = pd.concat([already_done_part, slice_to_work_on])
        updated_df.to_csv(output_file_path, index=False)
    
    print(f"✓ Task {task_number} ({task_name}) completato!")

print("✓ Funzione call_model_llm definita")

✓ Funzione call_model_llm definita


In [ ]:
print("="*60)
print("TASK 1: JSON formatting")
print("="*60)

call_model_llm(model_name=MODEL_NAME, task_number=1, input_file_path=INPUT_FILE_PATH)

TASK 1: JSON formatting
✓ Created new output file: ../resources/sampled_reviews_task_1_json_only_qwen3_4b-instruct.csv
Processing Task 1 (json_only) with 500 reviews...


Task 1:   1%|█▏                                                                                                                                                                                                    | 3/500 [00:55<2:13:56, 16.17s/it]

In [ ]:
print("="*60)
print("TASK 2: JSON + Traduzione")
print("="*60)

call_model_llm(model_name=MODEL_NAME, task_number=2, input_file_path=INPUT_FILE_PATH)

In [ ]:
print("="*60)
print("TASK 3: JSON + Traduzione + Sentiment Analysis")
print("="*60)

call_model_llm(model_name=MODEL_NAME, task_number=3, input_file_path=INPUT_FILE_PATH)

In [ ]:
print("="*60)
print("TASK 4: JSON + Traduzione + Sentiment + NER (Completo)")
print("="*60)

call_model_llm(model_name=MODEL_NAME, task_number=4, input_file_path=INPUT_FILE_PATH)

In [ ]:
print("\n" + "="*60)
print("PERFORMANCE DEGRADATION ANALYSIS COMPLETATA!")
print("="*60)

print("\nFile di output generati:")
for task_num in range(1, 5):
    task_name = TASK_NAMES[task_num]
    output_file = f"../resources/sampled_reviews_task_{task_num}_{task_name}_{MODEL_NAME.replace(':', '_')}.csv"
    if os.path.exists(output_file):
        df = pd.read_csv(output_file)
        completed_rows = len(df[df['output'] != '$$$'])
        total_rows = len(df)
        print(f"  ✓ Task {task_num} ({task_name}): {output_file}")
        print(f"    Progress: {completed_rows}/{total_rows} reviews processed")
    else:
        print(f"  ✗ Task {task_num} ({task_name}): File non trovato")

print(f"\nModello utilizzato: {MODEL_NAME}")
print(f"File input: {INPUT_FILE_PATH}")

In [ ]:
import json

print("="*60)
print("ANALISI RAPIDA DEI RISULTATI")
print("="*60)

for task_num in range(1, 5):
    task_name = TASK_NAMES[task_num]
    output_file = f"../resources/sampled_reviews_task_{task_num}_{task_name}_{MODEL_NAME.replace(':', '_')}.csv"
    
    if os.path.exists(output_file):
        df = pd.read_csv(output_file)
        completed_rows = len(df[df['output'] != '$$$'])
            
        total_rows = len(df)
        completion_rate = (completed_rows / total_rows) * 100

        valid_json_count = 0
        broken_json_count = 0
        
        if completed_rows > 0:
            completed_outputs = df[df['output'] != '$']['output']
            
            for output in completed_outputs:
                if pd.isna(output) or output is None:
                    broken_json_count += 1
                    continue
                    
                try:
                    # Prova a fare il parsing del JSON
                    json.loads(output)
                    valid_json_count += 1
                except (json.JSONDecodeError, TypeError):
                    broken_json_count += 1
        
        print(f"\nTask {task_num} - {task_name}:")
        print(f"  Completamento: {completed_rows}/{total_rows} ({completion_rate:.1f}%)")
        
        if completed_rows > 0:
            json_valid_rate = (valid_json_count / completed_rows) * 100
            print(f"  JSON validi: {valid_json_count}/{completed_rows} ({json_valid_rate:.1f}%)")
            print(f"  JSON rotti: {broken_json_count}/{completed_rows} ({100-json_valid_rate:.1f}%)")
            
            # Mostra un esempio di output
            sample_output = df[df['output'] != '$]['output'].iloc[0] if completed_rows > 0 else "N/A"
            print(f"  Esempio output: {sample_output[:100]}...")
        else:
            print(f"  JSON validi: 0/0 (N/A)")
            print(f"  JSON rotti: 0/0 (N/A)")
    else:
        print(f"\nTask {task_num} - {task_name}: File non trovato")